# 06 — Whale Optimization Algorithm (WOA) Feature Selection

Binary WOA searches for a compact feature mask. This notebook uses the shared experiment pipeline so all optimizers receive the same data splits, classifiers, seeds, and evaluation rules.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "utils").exists():
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))

import random
import numpy as np


## Binary Whale Optimization Algorithm

In [ ]:
# -------------------------------------------------
# Binary Whale Optimization Algorithm (BWOA)
# -------------------------------------------------

import numpy as np


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def run_bwoa(
    obj_func,
    n_features,
    pop_size=30,
    iterations=50,
):

    # Initialize whales
    positions = np.random.randint(0, 2, (pop_size, n_features))

    fitness = np.array([obj_func(w) for w in positions])

    best_idx = np.argmin(fitness)
    best_position = positions[best_idx].copy()
    best_score = fitness[best_idx]

    convergence = []

    b = 1

    for t in range(iterations):

        a = 2 - 2 * (t / iterations)

        for i in range(pop_size):

            r1 = np.random.rand()
            r2 = np.random.rand()

            A = 2 * a * r1 - a
            C = 2 * r2

            p = np.random.rand()

            new_position = np.zeros(n_features)

            if p < 0.5:

                if abs(A) < 1:

                    D = np.abs(C * best_position - positions[i])
                    X = best_position - A * D

                else:

                    rand_idx = np.random.randint(pop_size)
                    rand_whale = positions[rand_idx]

                    D = np.abs(C * rand_whale - positions[i])
                    X = rand_whale - A * D

            else:

                l = np.random.uniform(-1, 1)

                D = np.abs(best_position - positions[i])

                X = (
                    D
                    * np.exp(b * l)
                    * np.cos(2 * np.pi * l)
                    + best_position
                )

            probs = sigmoid(X)

            new_position = (
                np.random.rand(n_features) < probs
            ).astype(int)

            # Prevent empty subset
            if new_position.sum() == 0:
                new_position[np.random.randint(n_features)] = 1

            positions[i] = new_position

        fitness = np.array([obj_func(w) for w in positions])

        idx = np.argmin(fitness)

        if fitness[idx] < best_score:
            best_score = fitness[idx]
            best_position = positions[idx].copy()

        convergence.append(best_score)

    return best_position, best_score, convergence

## Run the complete feature-selection experiment

The shared experiment runner supplies the configured datasets, classifiers,
optimizer seeds, population size, and iteration count. Feature selection uses
the validation set. The test set is evaluated only after the final mask has
been selected.


In [ ]:
from utils.experiments import run_feature_selector


def woa_runner(
    objective,
    n_features,
    pop_size,
    iterations,
):
    return run_bwoa(
        obj_func=objective,
        n_features=n_features,
        pop_size=pop_size,
        iterations=iterations,
    )


woa_results = run_feature_selector("WOA", woa_runner)
woa_results.tail()
